# Secure Hashing & Encryption — Notebook
This notebook includes examples for hashing methods, encryption and decryption.

## 1. Imports

In [ ]:
# Import the sys module to adjust the Python system path
import sys

# Import the os module to interact with the operating system
import os
os.environ["JAVA_HOME"] = "/home/ec2-user/anaconda3/envs/spark"

# Install libraries inside the *same environment* as the active kernel
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "bcrypt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "argon2-cffi"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "cryptography"])

# Append a path to the system path so that Python can import modules from the specified directory (relative path)
sys.path.append('../')

# Import the CONF, QA_summary and QA_action_plan variables from the script.QA module
from script.conf import *

# Import necessary packages
import hashlib
import base64
import bcrypt
from argon2 import PasswordHasher
from cryptography.fernet import Fernet
import pandas as pd

ph = PasswordHasher()

In [ ]:
# Try to create a new directory at the path HASH_PATH using os.mkdir()
# If the directory already exists, print a message stating that it does
try:
    os.mkdir(HASH_PATH)
    print("Create new folder {}".format(HASH_PATH))
except FileExistsError:
    print("Folder {} already exists".format(HASH_PATH))

## 2. Import data

In [ ]:
# 1) Read CSV file (make sure your CSV file has headers "msisdn", "datetime", "latitude", "longitude")
df = pd.read_csv(BASE_PATH+HASH_FILE_INPUT, dtype=str )

# Drop rows where lon or lat is null or '\N'
df = df[~df["longitude"].isin([None, "\\N"])]
df = df[~df["latitude"].isin([None, "\\N"])]
df = df.dropna(subset=["longitude", "latitude"])

# Replace any 'nan' strings (from NaN)
df["msisdn"] = df["msisdn"].replace("nan", "")

print(df.head())

## 3. Hashing Functions (SHA-256, SHA-3, BLAKE2, Salted SHA-256)

🔐 1. SHA-256

What it is:
SHA-256 (Secure Hash Algorithm 256-bit) is part of the SHA-2 family, designed by the U.S. National Security Agency (NSA) and standardized by NIST.

How it works: It takes an input string (like a subscriber ID) and computes a fixed 256-bit (64-character hex) fingerprint.Even a tiny input change (e.g., 1234 → 1235) dramatically changes the output.

Security properties
- Collision-resistant (extremely hard to find two inputs with the same output)
- Widely used (TLS, blockchain, file integrity)
- Deterministic (same input → same output)

Weaknesses
- Fast → vulnerable to brute-force attacks if used unsalted for passwords or subscriber IDs.
- Not designed to resist GPU cracking.

Use cases
- De-identifying telecom identifiers only when a salt is used
- Integrity checking
- Multi-party hashing (with shared salt)
________________________________________
🔐 2. SHA-3-256

What it is: SHA-3 is a newer hashing standard based on the Keccak algorithm, selected by NIST in 2015 after a global cryptographic competition.

How it works: Unlike SHA-2, SHA-3 uses a sponge construction, which absorbs input and squeezes out a hash value.

Security properties
- Extremely robust design
- Resistant to length-extension attacks
- Modern and mathematically elegant
- Lower chance of structural weaknesses compared to SHA-2

Weaknesses
- Not widely adopted in older systems
- Performance slower than SHA-256

Use cases
- When you want a modern alternative to SHA-256
- Research and systems that require highest long-term security confidence
________________________________________
🔐 3. BLAKE2b

What it is: 
BLAKE2 is a cryptographic hash designed to be faster than SHA-256 while providing stronger security guarantees.BLAKE2b is optimized for 64-bit systems (servers, desktops).

How it works: Uses a structure derived from the ChaCha stream cipher. Also supports salting and personalization natively.

Security properties:

- Stronger design margin than SHA-2
- Up to 2× faster
- No known vulnerabilities
- Allows built-in salt, keying, and context strings

Weaknesses
- Less widely standardized than SHA-2

Use cases
- High-performance hashing
- Embedded within cryptographic protocols
- Fast anonymization in telecom/MPD workflows
________________________________________
🔐 4. SHA-256 + random per-record salt

What it is: 
"salt" is a random string appended to each identifier before hashing.

Example:
hash = SHA256(salt || msisdn)
Each value gets its own unique salt.

Why it matters: Without salt, attackers can do:
- Rainbow table attacks
- Dictionary attacks
- Precomputation attacks
- Cross-database correlation attacks

Randomized per-record salting prevents all of these.

Security impact
- Makes the hash unique even if two users have same ID
- Prevents reverse lookup
- Salts can be stored alongside hashes (not a secret)

Weaknesses
- Breaks multi-operator joining, because salted hashes don’t match between datasets
(use a shared salt for MPC-style workflows)

Use cases
- Internal anonymization
- De-identification when no cross-dataset linking is needed
________________________________________
🔐 5. SHA-256 + Pepper
What it is:

SHA-256 with a pepper is a method where a secret key (pepper) is added to the input before hashing.
Unlike a salt, which is public and can be stored with each hash, a pepper is a secret value that must be kept hidden, similar to an encryption key.

Peppering is commonly used to strengthen hashing systems against database theft: an attacker who steals your hashes cannot recompute or verify guesses without the secret pepper.

How it works:

A pepper is typically a long random byte string stored securely in a key vault, environment variable, HSM, or secret manager.

Security properties

- Adds strong protection even if an attacker steals the entire hash database
- Prevents offline guessing attacks because the attacker cannot recompute hashes
- Deterministic as long as the same pepper is used
- Works as an additional security layer on top of salt

Weaknesses

- Pepper must be securely stored — losing it makes all hashes irrecoverable
- Pepper compromise allows an attacker to recompute hashes
- Not suitable for cross-organization matching unless the pepper is shared (which breaks the secrecy purpose)
- Introduces key-management complexity (rotation, backup, recovery)

Use cases:

- Strengthening hashing for internal anonymization workflows
- Protecting datasets where IDs are sensitive (e.g., MSISDN, IMSI, IMEI)
- Defending against large-scale offline guessing attacks
- Adding secrecy on top of SHA-256 + salt hashing

In [ ]:
def hash_sha256(x: str) -> str:
    return hashlib.sha256(x.encode()).hexdigest()

def hash_sha3(x: str) -> str:
    return hashlib.sha3_256(x.encode()).hexdigest()

def hash_blake2(x: str) -> str:
    return hashlib.blake2b(x.encode()).hexdigest()

def hash_sha256_salt(x: str, salt: bytes = None):
    if salt is None:
        salt = os.urandom(16)
    return hashlib.sha256(salt + x.encode()).hexdigest(), salt

def hash_sha256_pepper(x: str, pepper: bytes) -> str:
    """
    Computes SHA256(pepper || x)
    Pepper must be a secret key stored securely.
    """
    if isinstance(pepper, str):
        pepper = pepper.encode()

    return hashlib.sha256(pepper + x.encode()).hexdigest()

## 4. Key-Stretching: PBKDF2, bcrypt, Argon2

🔐 6. PBKDF2-HMAC
What it is:

A password-based key-derivation function (PBKDF2) using HMAC (usually SHA-256).
It slows down hashing intentionally.

How it works:
PBKDF2(input, salt, iterations) → stretched hash
Using 100k–600k iterations is typical today.

Security properties:
- Slows down brute force by orders of magnitude
- NIST-approved
- Standard in authentication systems
- Works everywhere (fast CPU support)

Weaknesses:
- Still too fast for GPU/ASIC cracking
- Needs many iterations for modern security

Use cases:
- Hashing sensitive identifiers
- Key derivation from passwords
- Stronger than SHA-256 alone
________________________________________

🔐 7. bcrypt
What it is:

A slow, adaptive hashing function based on the Blowfish cipher.

How it works
bcrypt stores:
- cost factor
- salt
- hash
Inside a single encoded string, e.g.:
$2b$12$abcdefghijklmnopqrstuvCwxyz12345678901234567890

Security properties
- Slow by design
- Automatically salts values
- Resistant to rainbow tables
- Common in authentication systems

Weaknesses
- Max input size is limited (72 bytes) — but MSISDNs are short
- Vulnerable to modern GPUs (but still much safer than SHA-256)

Use cases
- Protecting identifiers in environments where cracking risk is high
- Local reversible (with encryption) + irreversible (bcrypt) anonymization
________________________________________

🔐 8. Argon2 (Argon2id)

What it is:
Argon2 won the Password Hashing Competition (PHC) and is designed specifically to stop GPU and ASIC attackers.

Variants:
- Argon2d — GPU resistant
- Argon2i — side-channel safe
- Argon2id — hybrid and the recommended default

How it works
Argon2 uses:
- Memory hardness (e.g. 256 MB)
- Time cost (iterations)
- Parallelism
This makes attacks extremely expensive.

Security properties
- Most secure mainstream hashing function
- Memory-hard → GPUs lose advantage
- Adaptive (tune memory/time parameters)
- Built-in salts

Weaknesses
- Slower than PBKDF2
- Needs memory configuration
- Requires "argon2-cffi" package

Use cases
- Highest-security telecom/MPD anonymization
- Data that must remain safe for decades
- Protection against nation-state attackers

In [ ]:
def hash_pbkdf2(x: str, iterations=200000):
    salt = os.urandom(16)
    dk = hashlib.pbkdf2_hmac('sha256', x.encode(), salt, iterations)
    return base64.b64encode(dk).decode(), salt, iterations

def hash_bcrypt(x: str):
    return bcrypt.hashpw(x.encode(), bcrypt.gensalt()).decode()

def check_bcrypt(x, hashed):
    return bcrypt.checkpw(x.encode(), hashed.encode())

def hash_argon2(x: str) -> str:
    return ph.hash(x)

def verify_argon2(x: str, hashed: str) -> bool:
    return ph.verify(hashed, x)

## 5. AES-256 Encryption & Decryption Utilities

In [ ]:
def generate_key(path='secret.key'):
    key = Fernet.generate_key()
    with open(path,'wb') as f: f.write(key)
    return key

def load_key(path='secret.key'):
    with open(path,'rb') as f: return f.read()

def encrypt_value(x: str, cipher: Fernet) -> str:
    return cipher.encrypt(x.encode()).decode()

def decrypt_value(x: str, cipher: Fernet) -> str:
    return cipher.decrypt(x.encode()).decode()

## 6. Column-Level Encryption/Decryption

In [ ]:
def encrypt_column(df, column, cipher):
    df[column + '_enc'] = df[column].astype(str).apply(lambda x: encrypt_value(x, cipher))
    return df

def decrypt_column(df, column, cipher):
    df[column + '_dec'] = df[column].astype(str).apply(lambda x: decrypt_value(x, cipher))
    return df

## 7. File-Level Encryption

In [ ]:
# def encrypt_file(input_path, output_path, cipher):
#     with open(input_path, 'rb') as f:
#         data = f.read()
#     with open(output_path, 'wb') as f:
#         f.write(cipher.encrypt(data))

# def decrypt_file(input_path, output_path, cipher):
#     with open(input_path, 'rb') as f:
#         data = f.read()
#     with open(output_path, 'wb') as f:
#         f.write(cipher.decrypt(data))

## 8. Secure Multi-Party Hashing Workflow (Telecom/MPD)

In [ ]:
# # Shared salt securely exchanged between operators
# SALT = b'example_shared_secret_salt'

# def operator_hash(x: str):
#     return hashlib.sha256(SALT + x.encode()).hexdigest()

## 9. Example

In [ ]:
df['sha256'] = df['msisdn'].apply(hash_sha256)
print('sha256 done')
df['sha3'] = df['msisdn'].apply(hash_sha3)
print('sha3 done')
df['blake2'] = df['msisdn'].apply(hash_blake2)
print('blake2 done')
df['sha256_salt'] = df['msisdn'].apply(hash_sha256_salt)
print('sha256_salt done')

PEPPER = os.urandom(32)   # Or load from a secret vault
df["sha256_pepper"] = df["msisdn"].astype(str).apply(lambda x: hash_sha256_pepper(x, PEPPER))
print('sha256_pepper done')

df['argon2'] = df['msisdn'].apply(hash_argon2)
print('argon2 done')
df['pbkdf2'] = df['msisdn'].apply(hash_pbkdf2)
print('pbkdf2 done')
df['bcrypt'] = df['msisdn'].apply(hash_bcrypt)
print('bcrypt done')

key = Fernet.generate_key()
cipher = Fernet(key)

df['subscription_enc'] = df['msisdn'].apply(lambda x: encrypt_value(x, cipher))
df['subscription_dec'] = df['subscription_enc'].apply(lambda x: decrypt_value(x, cipher))

print(df.head())

In [ ]:
# Save the file
df.to_csv(HASH_PATH+HASH_FILE_OUTPUT, index=False)